In [1]:
!pip install gradio groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.0 MB/s eta 0:00:00


In [2]:
import os
os.environ["GROQ_API_KEY"] = "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"  # ← paste Groq key here
print("✅ Groq API Key set!")

✅ Groq API Key set!


In [3]:
import os, json, uuid, random, datetime, hashlib
from pathlib import Path
import gradio as gr
import pandas as pd
from groq import Groq

# ── CONFIG ──────────────────────────────────────────────────────────────────
GROQ_KEY = os.environ.get("GROQ_API_KEY", "")
groq_client = Groq(api_key=GROQ_KEY)

# ── DATA STORAGE ─────────────────────────────────────────────────────────────
Path("data").mkdir(exist_ok=True)
for f in ["bookings.json", "wishlists.json", "users.json"]:
    if not Path(f"data/{f}").exists():
        Path(f"data/{f}").write_text("{}")

def load_json(f): return json.loads(Path(f"data/{f}").read_text())
def save_json(f, d): Path(f"data/{f}").write_text(json.dumps(d, indent=2))

# ── CORE AI FUNCTION ─────────────────────────────────────────────────────────
def ask_ai(prompt, use_search=True):
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": """You are WanderAI, a world-class AI travel expert.
Always use INR as primary currency. Format responses with emojis.
Give real hotel names, real restaurant names, realistic prices."""},
                {"role": "user", "content": prompt}
            ],
            max_tokens=2048,
            temperature=0.8
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Groq Error: {str(e)}"

# ── DYNAMIC AI FUNCTIONS ─────────────────────────────────────────────────────

def search_destinations_ai(query, dtype, budget, season):
    prompt = f"""Search and list 6 real travel destinations matching:
Query: {query or 'popular destinations'}
Type: {dtype}
Budget: {budget} INR per day per person
Season: {season}

For each destination provide REAL current information:
**[Destination Name], [Country]**
- 📍 Region & Type
- ⭐ Rating (from real reviews)
- 💰 Budget: Rs.X,XXX - Rs.XX,XXX per day
- 🌤️ Best months to visit
- 🎯 Top 3 attractions (real names)
- 🍽️ Must-try foods (real dishes)
- ✈️ How to reach from India
- 📝 Brief description (2-3 lines)

Separate each destination with ---"""
    return ask_ai(prompt)

def search_hotels_ai(destination, budget, stars, checkin, checkout):
    nights = 1
    try:
        ci = datetime.date.fromisoformat(checkin)
        co = datetime.date.fromisoformat(checkout)
        nights = max((co - ci).days, 1)
    except: pass

    prompt = f"""Find 5 REAL hotels in {destination} with current prices:
Budget: Up to Rs.{budget:,} per night
Star Rating: {stars}+ stars
Check-in: {checkin} | Check-out: {checkout} ({nights} nights)

For each hotel provide REAL information:
**[Hotel Name]** ⭐⭐⭐⭐⭐
- 📍 Address (real area/locality)
- 💰 Price: Rs.X,XXX - Rs.XX,XXX per night (current rates)
- ⭐ Rating: X.X/5 (from real guest reviews)
- 🛎️ Top amenities (pool, spa, restaurant etc)
- 🔑 Check-in: X PM | Check-out: X AM
- 🔄 Cancellation policy
- 💡 Why stay here (unique selling point)
- 📱 Booking: Available on MakeMyTrip/Booking.com/Hotels.com

Separate each hotel with ---
Note: Provide ACTUAL current market prices for {datetime.date.today().strftime('%B %Y')}"""
    return ask_ai(prompt)

def search_flights_ai(origin, destination, travel_date, flight_class):
    prompt = f"""Find REAL flight options:
Route: {origin} → {destination}
Date: {travel_date or 'Next month (flexible)'}
Class: {flight_class}

List 5 real airlines operating this route with current prices:
**[Airline Name]** - Flight [Number]
- ⏰ Departure: XX:XX | Arrival: XX:XX
- ⏱️ Duration: Xh Xm | Stops: Non-stop/1 stop
- 💰 Price: Rs.X,XXX - Rs.XX,XXX per person ({flight_class})
- 🧳 Baggage: Xkg included
- 🍽️ Meal: Included/Not included
- 🔄 Cancellation: Policy details
- 📱 Book on: IndiGo/MakeMyTrip/Cleartrip/Goibibo

Also mention:
- 💡 Cheapest booking tip for this route
- 📅 Best days to fly for lowest price
- ⚡ Current demand (high/medium/low season)

Separate each flight with ---"""
    return ask_ai(prompt)

def search_packages_ai(destination, budget, style, days):
    prompt = f"""Create 4 REAL travel packages for:
Destination: {destination or 'Popular Indian/International destinations'}
Budget: Up to Rs.{budget:,} per person
Style: {style}
Duration: {days} days

For each package provide:
**[Package Name]** 🔥
- 📍 Destination | ⏱️ Duration: X Days/X Nights
- 💰 Price: Rs.XX,XXX per person (was Rs.XX,XXX - XX% off)
- ⭐ Rating: X.X | 👥 Group size
- 📅 Availability: Months

✅ INCLUSIONS (list real inclusions):
- Flights from [city]
- Hotel name and category
- Meals included
- Activities and tours

❌ EXCLUSIONS:
- What's not included

🎯 HIGHLIGHTS:
- 3 unique experiences

🏨 Suggested Hotels: [Real hotel names]
📱 Book on: MakeMyTrip/Thomas Cook/SOTC/Yatra

Separate each package with ---"""
    return ask_ai(prompt)

def generate_trip_plan_ai(destination, budget, days, travelers, style, start_date):
    prompt = f"""Create a DETAILED real-world trip plan:

🌍 Destination: {destination}
💰 Total Budget: Rs.{budget:,} (Rs.{budget//travelers:,} per person)
📅 Duration: {days} days | 👥 Travelers: {travelers}
🎭 Style: {style} | 🗓️ Start: {start_date or 'Flexible'}

Provide a COMPLETE, ACTIONABLE trip plan:

## ✈️ HOW TO REACH
- Best flights from major Indian cities (with real airlines and price ranges)
- Airport/transport to hotel

## 🗓️ DAY-BY-DAY ITINERARY
For EACH day:
**Day X — [Theme]**
- 🌅 Morning (7AM-12PM): [Specific activity + real venue name + cost]
- ☀️ Afternoon (12PM-6PM): [Specific activity + real venue + cost]
- 🌙 Evening (6PM-10PM): [Restaurant name + what to eat + cost]
- 🚗 Transport between locations
- 💰 Estimated day spend: Rs.XXX

## 🏨 WHERE TO STAY
3 real hotel options with current prices:
- Budget: [Hotel name] - Rs.X,XXX/night
- Mid-range: [Hotel name] - Rs.X,XXX/night
- Luxury: [Hotel name] - Rs.XX,XXX/night

## 💰 COMPLETE BUDGET BREAKDOWN
| Category | Amount (Rs.) | % of Budget |
|----------|-------------|-------------|
| Flights | | |
| Hotel ({days-1} nights) | | |
| Food | | |
| Activities | | |
| Transport (local) | | |
| Shopping/Misc | | |
| **TOTAL** | | |

## 🎯 TOP EXPERIENCES FOR {style.upper()} TRAVELER
5 specific activities with real costs and booking tips

## 💡 INSIDER TIPS
- Best time to visit specific attractions
- How to save money
- What to avoid
- Local customs"""
    return ask_ai(prompt)

def generate_itinerary_ai(destination, days, style, interests, budget_level):
    prompt = f"""Generate a MINUTE-BY-MINUTE {days}-day itinerary for {destination}:
Style: {style} | Interests: {interests} | Budget: {budget_level}

For EACH of the {days} days:

**Day X — [Catchy Theme Title]**

🌅 **MORNING** (6:00 AM - 12:00 PM)
- 6:00 AM: [Specific activity] at [Real venue name]
- 8:00 AM: Breakfast at [Real restaurant] - must try [dish] (Rs.XXX)
- 9:30 AM: [Activity] - [insider tip]
- 11:00 AM: [Next activity]

☀️ **AFTERNOON** (12:00 PM - 6:00 PM)
- 12:30 PM: Lunch at [Real restaurant] (Rs.XXX-XXX)
- 2:00 PM: [Activity] at [Real venue]
- 4:00 PM: [Activity]
- 5:30 PM: [Photography spot / relaxation]

🌙 **EVENING** (6:00 PM - 10:00 PM)
- 6:00 PM: [Sunset spot with real name]
- 7:30 PM: Dinner at [Real restaurant] - specialty [dish] (Rs.XXX)
- 9:00 PM: [Evening activity]

🚗 **TRANSPORT TODAY**: [How to get around]
💰 **DAILY BUDGET**: Rs.X,XXX - Rs.X,XXX ({budget_level})
🏨 **STAY**: [Hotel area recommendation]
💡 **PRO TIP**: [One insider secret for this day]

Include real venue names, actual prices, and genuine local recommendations!"""
    return ask_ai(prompt)

def recommend_destinations_ai(interests, budget, season, travelers, style):
    prompt = f"""Recommend TOP 6 destinations based on:
Interests: {interests}
Budget: {budget}
Travel Season: {season}
Group: {travelers}
Style: {style}

For each destination:

## 🌍 [Rank]. [Destination Name], [Country]
**Why it's PERFECT for you:** [Personalized reason based on interests]

📊 **Quick Stats:**
- ⭐ Rating: X.X/5
- 💰 Budget: Rs.X,XXX - Rs.XX,XXX per day
- 🗓️ Best months: [Months]
- 🌤️ Weather in {season}: [Description]

✈️ **Getting There from India:**
- Flight from Delhi/Mumbai: Rs.X,XXX - Rs.XX,XXX
- Flight duration: Xh Xm
- Visa: [Required/Not required + details]

🎯 **Top 5 Experiences:**
1. [Real experience with cost]
2. [Real experience with cost]
3. [Real experience with cost]
4. [Real experience with cost]
5. [Real experience with cost]

🏨 **Where to Stay:**
- Budget: [Hotel name] Rs.X,XXX/night
- Luxury: [Hotel name] Rs.XX,XXX/night

🍽️ **Must Try:** [3 real local dishes]

⚠️ **Watch Out For:** [1-2 honest warnings]

Separate each with ═══════════════════"""
    return ask_ai(prompt)

def optimize_budget_ai(destination, total_budget, days, travelers):
    per_day = total_budget // days
    per_person = total_budget // travelers
    prompt = f"""Create a SMART budget plan with real prices:

Destination: {destination}
Total Budget: Rs.{total_budget:,}
Duration: {days} days | Travelers: {travelers}
Per Day: Rs.{per_day:,} | Per Person: Rs.{per_person:,}

## 💰 OPTIMAL BUDGET ALLOCATION
| Category | Amount | % | Tips |
|----------|--------|---|------|
| Flights | Rs.X,XXX | XX% | [Booking tip] |
| Accommodation | Rs.X,XXX | XX% | [Value pick] |
| Food | Rs.X,XXX | XX% | [Strategy] |
| Activities | Rs.X,XXX | XX% | [Free vs paid] |
| Local Transport | Rs.X,XXX | XX% | [Best option] |
| Shopping | Rs.X,XXX | XX% | [What to buy] |
| Emergency | Rs.X,XXX | XX% | [Keep aside] |

## 🏨 BEST VALUE HOTELS IN {destination.upper()}
1. **[Hotel Name]** - Rs.X,XXX/night - [Why good value]
2. **[Hotel Name]** - Rs.X,XXX/night - [Why good value]
3. **[Hotel Name]** - Rs.X,XXX/night - [Why good value]

## ✈️ FLIGHT STRATEGY
- Best airline for this route: [Real airline + price]
- Book [X] days in advance for best price
- Cheapest day to fly: [Day]
- Use: [Booking platform] for best deals

## 🍽️ SMART FOOD STRATEGY
- Breakfast: Rs.XXX/day at [type of place]
- Lunch: Rs.XXX/day at [type of place]
- Dinner: Rs.XXX/day at [type of place]
- Street food budget: Rs.XXX/day

## 🎯 FREE & LOW-COST ACTIVITIES
List 5 free or under Rs.500 experiences in {destination}

## 💡 TOP 10 MONEY HACKS for {destination}
Specific, actionable tips to save money

## 📊 FINAL VERDICT
Can you do {days} days in {destination} for Rs.{total_budget:,}?
What will you have to compromise vs what you'll enjoy?"""
    return ask_ai(prompt)

def chat_ai(message, history):
    context = ""
    if history:
        context = "\n".join([f"User: {h[0]}\nAssistant: {h[1][:200]}..."
                             for h in history[-3:]])
    prompt = f"""You are WanderAI travel assistant.
Previous conversation:
{context}

User question: {message}

Provide a helpful, specific, and accurate travel response.
Include real prices, real place names, and practical advice.
Use INR for prices. Be conversational but informative."""
    response = ask_ai(prompt)
    history.append([message, response])
    return history, ""

def crowd_prediction_ai(destination, travel_date):
    prompt = f"""Provide REAL crowd prediction for {destination} in {travel_date}:

## 📊 CROWD LEVEL FORECAST
Overall Rating: [X]/10 (🟢 Low / 🟡 Medium / 🔴 High)

## 📅 WEEK-BY-WEEK BREAKDOWN
Week 1: Crowd level + reason
Week 2: Crowd level + reason
Week 3: Crowd level + reason
Week 4: Crowd level + reason

## 🏛️ ATTRACTION-SPECIFIC CROWDS
For top 5 attractions in {destination}:
- [Attraction]: Best time to visit, expected wait time, booking tip

## ✅ BEAT THE CROWDS STRATEGY
- Best time of day to visit popular spots
- Which days to avoid
- Lesser-known alternatives

## 🏨 ACCOMMODATION DEMAND
- Hotel prices in {travel_date}: Rs.X,XXX - Rs.XX,XXX
- Book how far in advance: X weeks/months

## 🎯 FINAL RECOMMENDATION
🟢 GREAT TIME / 🟡 AVERAGE / 🔴 AVOID — with explanation"""
    return ask_ai(prompt)

def weather_ai(destination, month):
    prompt = f"""Real weather guide for {destination} in {month}:

## 🌡️ TEMPERATURE
- Average High: XX°C | Average Low: XX°C
- Humidity: XX%

## 🌧️ RAINFALL
- Rainfall: XXmm | Rainy days: X days

## ☀️ SUNSHINE
- Daily sunshine: X hours | UV Index: X

## 👗 PACKING LIST
- Clothing, Footwear, Accessories for this weather

## 🎯 BEST ACTIVITIES FOR THIS WEATHER
What's GREAT and what's NOT recommended in {month}

## ⚡ WEATHER RISKS
Any cyclones, heat waves, floods history for {month}

## 📊 COMPARED TO INDIA
Context for Indian travelers"""
    return ask_ai(prompt)

def safety_ai(destination, travel_date):
    prompt = f"""Real travel safety report for {destination} in {travel_date} for Indian travelers:

## 🛡️ OVERALL SAFETY RATING: X.X/10

## 🚦 CATEGORY RATINGS
| Category | Rating | Details |
|----------|--------|---------|
| Personal Safety | X/10 | |
| Health & Medical | X/10 | |
| Political Stability | X/10 | |
| Natural Disaster Risk | X/10 | |
| Scam Risk | X/10 | |

## 🏥 HEALTH
- Required vaccinations, tap water safety, nearest hospital

## 🚔 TOP 5 SCAMS targeting tourists with how to avoid them

## 🛂 ENTRY REQUIREMENTS FOR INDIANS
- Visa details, cost, passport validity required

## 📱 EMERGENCY NUMBERS
Police, Ambulance, Indian Embassy, Tourist Helpline

## ✅ OVERALL VERDICT
Is it safe to travel to {destination} in {travel_date}?"""
    return ask_ai(prompt)

def cultural_guide_ai(destination):
    prompt = f"""Complete cultural etiquette guide for {destination} for Indian travelers:

## 🙏 GREETINGS & SOCIAL CUSTOMS
## 👗 DRESS CODE (streets, temples, beaches, restaurants)
## 🍽️ DINING ETIQUETTE (tipping, eating customs)
## 📸 PHOTOGRAPHY RULES (what's allowed, what's prohibited)
## 🛒 BARGAINING CULTURE (where to bargain, how much discount)
## 🚫 ABSOLUTE TABOOS (top 5 things that will offend locals)
## 💬 USEFUL PHRASES
| English | Local Language | Pronunciation |
|---------|---------------|---------------|
| Hello | | |
| Thank you | | |
| How much? | | |
| Where is? | | |
## ⚠️ COMMON TOURIST MISTAKES Indians make in {destination}"""
    return ask_ai(prompt)

def food_guide_ai(destination):
    prompt = f"""Ultimate REAL food guide for {destination}:

## 🌟 10 DISHES YOU MUST TRY
For each: dish name, price range, where to find it, what it is, ordering tip

## 🏆 ICONIC RESTAURANTS
- Fine Dining (Rs.X,XXX+): [Real restaurant names]
- Mid-range (Rs.XXX-X,XXX): [Real restaurant names]
- Street Food (under Rs.XXX): [Real stall/area names]

## 🌅 BREAKFAST CULTURE
What locals eat, where, timing, cost

## 🛒 FOOD MARKETS
Real market names, what to buy, best time to visit

## 🥂 LOCAL DRINKS
Non-alcoholic and alcoholic specialties

## 🎁 EDIBLE SOUVENIRS to bring back to India

## ⚠️ FOOD SAFETY TIPS

## 💰 FOOD BUDGET GUIDE
| Meal | Budget | Mid | Splurge |
|------|--------|-----|---------|
| Breakfast | | | |
| Lunch | | | |
| Dinner | | | |"""
    return ask_ai(prompt)

def book_hotel_ai(hotel_name, destination, checkin, checkout, rooms, guests, name, email):
    try:
        ci = datetime.date.fromisoformat(checkin)
        co = datetime.date.fromisoformat(checkout)
        nights = max((co - ci).days, 1)
    except:
        nights = 1

    bid = f"WA{datetime.date.today().strftime('%Y%m')}{random.randint(10000,99999)}"
    pay_id = f"PAY{random.randint(100000,999999)}"

    price_prompt = f"What is the approximate price per night in INR for {hotel_name} in {destination}? Reply with just a number, no text."
    price_text = ask_ai(price_prompt)
    try:
        import re
        numbers = re.findall(r'\d+', price_text.replace(',',''))
        price_per_night = int(numbers[0]) if numbers else 8000
    except:
        price_per_night = 8000

    base = price_per_night * nights * rooms
    gst = base * 0.18
    fee = base * 0.05
    total = base + gst + fee

    booking = {
        "id": bid, "type": "Hotel", "hotel": hotel_name, "destination": destination,
        "user_name": name, "user_email": email, "rooms": rooms, "guests": guests,
        "check_in": checkin, "check_out": checkout, "nights": nights,
        "price_per_night": price_per_night, "total_price_inr": total,
        "status": "CONFIRMED", "created_at": datetime.datetime.now().isoformat(),
        "payment_id": pay_id
    }

    bookings = load_json("bookings.json")
    bookings[bid] = booking
    save_json("bookings.json", bookings)

    receipt = f"""
╔══════════════════════════════════════════════════════╗
║          🌍 WanderAI — BOOKING CONFIRMED!           ║
╠══════════════════════════════════════════════════════╣
║  Booking ID  : {bid:<36}║
║  Payment ID  : {pay_id:<36}║
║  Status      : ✅ CONFIRMED                         ║
╠══════════════════════════════════════════════════════╣
║  HOTEL DETAILS                                      ║
║  Hotel       : {hotel_name[:36]:<36}║
║  Destination : {destination[:36]:<36}║
║  Guest       : {name[:36]:<36}║
║  Rooms       : {str(rooms):<36}║
║  Guests      : {str(guests):<36}║
║  Check-in    : {checkin:<36}║
║  Check-out   : {checkout:<36}║
║  Nights      : {str(nights):<36}║
╠══════════════════════════════════════════════════════╣
║  PAYMENT BREAKDOWN                                  ║
║  Rate/Night  : Rs.{price_per_night:>10,.0f}{' '*22}║
║  Base Amount : Rs.{base:>10,.0f}{' '*22}║
║  GST (18%)   : Rs.{gst:>10,.0f}{' '*22}║
║  Service Fee : Rs.{fee:>10,.0f}{' '*22}║
║  TOTAL       : Rs.{total:>10,.0f}{' '*22}║
╠══════════════════════════════════════════════════════╣
║  Free cancellation up to 48 hours before check-in  ║
║  support@wanderai.com  |  1800-WANDER              ║
╚══════════════════════════════════════════════════════╝
✅ Confirmation sent to: {email}
🎉 Have an amazing trip! Safe travels!
    """
    return receipt

def get_bookings(email):
    bookings = load_json("bookings.json")
    user_bookings = [b for b in bookings.values()
                     if b.get("user_email","").lower() == email.lower()]
    if not user_bookings:
        return "<p style='color:#718096;text-align:center'>No bookings found.</p>"
    html = f"<p style='font-weight:600;color:#553C9A'>Found {len(user_bookings)} booking(s)</p>"
    for b in sorted(user_bookings, key=lambda x: x.get("created_at",""), reverse=True):
        html += f"""<div style="border:1px solid #E2E8F0;border-radius:10px;padding:14px;margin:8px 0;background:white">
<div style="display:flex;justify-content:space-between">
<div>
<p style="margin:0;font-weight:600">🎫 {b['id']}</p>
<p style="margin:2px 0;color:#718096;font-size:13px">{b.get('hotel','Package')} | {b.get('destination','')}</p>
<p style="margin:2px 0;color:#718096;font-size:13px">{b['check_in']} → {b['check_out']}</p>
<p style="margin:2px 0;color:#A0AEC0;font-size:11px">Pay ID: {b.get('payment_id','')}</p>
</div>
<div style="text-align:right">
<p style="color:#38A169;font-weight:600">✅ {b['status']}</p>
<p style="font-size:18px;font-weight:700;color:#0F4C81">Rs.{b['total_price_inr']:,.0f}</p>
</div>
</div>
</div>"""
    return html

def add_wishlist(email, item_type, item_name):
    if not email.strip() or not item_name.strip():
        return "<div style='color:red'>❌ Enter email and item name!</div>"
    wl = load_json("wishlists.json")
    if email not in wl: wl[email] = []
    item = {"type": item_type, "name": item_name,
            "added_at": datetime.datetime.now().isoformat()}
    wl[email].append(item)
    save_json("wishlists.json", wl)
    return f"<div style='color:green'>✅ '{item_name}' added to wishlist! ❤️</div>"

def view_wishlist(email):
    if not email.strip(): return "<div style='color:red'>❌ Enter email!</div>"
    wl = load_json("wishlists.json")
    items = wl.get(email, [])
    if not items: return "<p style='color:#718096'>Wishlist empty! Start adding destinations ❤️</p>"
    colors = {"Destination":"#3182CE","Hotel":"#38A169","Package":"#E53E3E","Activity":"#805AD5"}
    html = f"<p style='font-weight:600;color:#C53030'>❤️ Your Wishlist ({len(items)} items)</p>"
    for x in items:
        c = colors.get(x['type'],'#718096')
        html += f"""<div style="border:1px solid #FED7D7;border-radius:8px;padding:10px;margin:6px 0;background:white;display:flex;justify-content:space-between;align-items:center">
<div>
<span style="background:{c};color:white;padding:2px 7px;border-radius:12px;font-size:11px">{x['type']}</span>
<span style="margin-left:8px;font-weight:500">{x['name']}</span>
<p style="margin:2px 0;color:#A0AEC0;font-size:11px">Added: {x.get('added_at','')[:10]}</p>
</div>
<span style="color:#E53E3E;font-size:20px">❤️</span>
</div>"""
    return html

def create_account(name, email, password):
    if not all([name.strip(), email.strip(), password.strip()]):
        return "<div style='color:red'>❌ Fill all fields!</div>"
    users = load_json("users.json")
    if email in users: return "<div style='color:red'>❌ Email already registered!</div>"
    uid = str(uuid.uuid4())[:8].upper()
    users[email] = {"id": uid, "name": name, "email": email,
                    "password": hashlib.sha256(password.encode()).hexdigest(),
                    "created_at": datetime.datetime.now().isoformat()}
    save_json("users.json", users)
    return f"<div style='color:green'>✅ Welcome {name}! Account created. ID: {uid}</div>"

def login(email, password):
    users = load_json("users.json")
    if email not in users: return "<div style='color:red'>❌ Email not found!</div>", ""
    if users[email]["password"] != hashlib.sha256(password.encode()).hexdigest():
        return "<div style='color:red'>❌ Wrong password!</div>", ""
    u = users[email]
    profile = f"""<div style="background:#EBF4FF;border-radius:10px;padding:16px">
<h3 style="margin:0;color:#3730A3">👋 Welcome back, {u['name']}!</h3>
<p style="color:#4C51BF">User ID: {u['id']} | Member since: {u['created_at'][:10]}</p>
<p style="color:#4A5568">🌍 Ready to plan your next adventure?</p>
</div>"""
    return "<div style='color:green'>✅ Login successful!</div>", profile

# ── BUILD GRADIO UI ──────────────────────────────────────────────────────────
css = """
.gradio-container { max-width:1200px !important; font-family:'Segoe UI',system-ui,sans-serif !important; }
.tab-nav button { font-size:13px !important; font-weight:500 !important; }
.tab-nav button.selected { background:#0F4C81 !important; color:white !important; font-weight:600 !important; }
"""

with gr.Blocks(title="🌍 WanderAI — AI Tourism Platform", css=css,
               theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.HTML("""<div style="background:linear-gradient(135deg,#0F4C81,#1a6eb8,#0F4C81);border-radius:16px;padding:28px;text-align:center;margin-bottom:16px;box-shadow:0 8px 32px rgba(15,76,129,0.3)">
  <h1 style="margin:0;color:white;font-size:2.4em;font-weight:800">🌍 WanderAI</h1>
  <p style="margin:8px 0 0;color:rgba(255,255,255,0.9);font-size:15px">Real-Time AI Travel Platform — Powered by Groq LLaMA 3.3</p>
  <div style="margin-top:12px;display:flex;justify-content:center;gap:12px;flex-wrap:wrap">
    <span style="background:rgba(255,255,255,0.2);color:white;padding:4px 14px;border-radius:20px;font-size:12px">⚡ Groq Powered</span>
    <span style="background:rgba(255,255,255,0.2);color:white;padding:4px 14px;border-radius:20px;font-size:12px">🦙 LLaMA 3.3 70B</span>
    <span style="background:rgba(255,255,255,0.2);color:white;padding:4px 14px;border-radius:20px;font-size:12px">✈️ Any Destination</span>
    <span style="background:rgba(255,255,255,0.2);color:white;padding:4px 14px;border-radius:20px;font-size:12px">♾️ Unlimited Results</span>
  </div>
</div>""")

    with gr.Tabs():

        # ── TAB 1: TRIP PLANNER ────────────────────────────────────────────
        with gr.Tab("🗓️ Trip Planner"):
            gr.HTML('<div style="background:#EBF8FF;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #0F4C81"><h3 style="margin:0;color:#0F4C81">🤖 AI Trip Planner — Any Destination Worldwide</h3><p style="margin:4px 0 0;color:#4A5568">Enter ANY destination — AI will create a complete real-time travel plan!</p></div>')
            with gr.Row():
                with gr.Column(scale=1):
                    tp_dest  = gr.Textbox(label="🌍 Destination", placeholder="Anywhere! e.g. Santorini, Kyoto, Ladakh, Paris...", value="Goa")
                    with gr.Row():
                        tp_budget = gr.Number(label="💰 Total Budget (Rs.)", value=50000)
                        tp_trav   = gr.Number(label="👥 Travelers", value=2)
                    with gr.Row():
                        tp_days  = gr.Slider(label="🗓️ Days", minimum=1, maximum=30, value=5, step=1)
                        tp_style = gr.Dropdown(label="🎭 Style", value="Relaxation",
                                               choices=["Adventure","Relaxation","Luxury","Cultural","Romantic","Family","Budget","Foodie","Spiritual","Backpacking"])
                    tp_date  = gr.Textbox(label="📅 Travel Month", placeholder="e.g. December 2025")
                    tp_btn   = gr.Button("🚀 Generate Trip Plan", variant="primary", size="lg")
                with gr.Column(scale=2):
                    tp_out = gr.Markdown(value="*Enter any destination and click Generate!*")
            tp_btn.click(fn=lambda d,b,t,dy,s,dt: generate_trip_plan_ai(d,int(b),int(dy),int(t),s,dt),
                        inputs=[tp_dest,tp_budget,tp_trav,tp_days,tp_style,tp_date], outputs=tp_out)

        # ── TAB 2: DESTINATIONS ────────────────────────────────────────────
        with gr.Tab("🌍 Destinations"):
            gr.HTML('<div style="background:#FFF5F0;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #FF6B35"><h3 style="margin:0;color:#C05621">🌏 AI Destination Search — Search ANYWHERE!</h3></div>')
            with gr.Row():
                ds_query  = gr.Textbox(label="🔍 Search", placeholder="e.g. hidden gems Europe, budget beach destinations, mountains near Delhi...", scale=3)
                ds_type   = gr.Dropdown(label="Type", choices=["Any","Beach","Mountains","Cultural","Adventure","Luxury","Budget","Food","Nature","Spiritual","Urban"], value="Any", scale=1)
            with gr.Row():
                ds_budget = gr.Number(label="💰 Max Budget/Day (Rs.)", value=5000)
                ds_season = gr.Dropdown(label="🗓️ Season", choices=["Any Season","Winter (Oct-Feb)","Summer (Mar-Jun)","Monsoon (Jul-Sep)"], value="Any Season")
                ds_btn    = gr.Button("🔍 Search", variant="primary", scale=1)
            ds_out = gr.Markdown(value="*Search any destination — AI finds real matches with current info!*")
            ds_btn.click(fn=lambda q,t,b,s: search_destinations_ai(q,t,int(b) if b else 0,s),
                        inputs=[ds_query,ds_type,ds_budget,ds_season], outputs=ds_out)

            with gr.Accordion("🤖 AI Destination Guide", open=False):
                dg_dest = gr.Textbox(label="Destination", placeholder="Enter ANY destination...", value="Kyoto, Japan")
                dg_btn  = gr.Button("🤖 Generate Guide", variant="primary")
                dg_out  = gr.Markdown()
                dg_btn.click(fn=lambda d: ask_ai(f"Write a detailed travel guide for {d}. Cover top attractions with real names, food scene with actual restaurants, shopping, nightlife, getting around, insider tips and best photo spots. Include real prices in INR."), inputs=dg_dest, outputs=dg_out)

        # ── TAB 3: HOTELS ──────────────────────────────────────────────────
        with gr.Tab("🏨 Find Hotels"):
            gr.HTML('<div style="background:#F0FFF4;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #38A169"><h3 style="margin:0;color:#276749">🏨 AI Hotel Search — Any City Worldwide</h3></div>')
            with gr.Row():
                ht_dest  = gr.Textbox(label="🌍 City", placeholder="e.g. Udaipur, Amsterdam, New York...", value="Goa")
                ht_price = gr.Number(label="💰 Max Price/Night (Rs.)", value=15000)
            with gr.Row():
                ht_stars = gr.Dropdown(label="⭐ Stars", choices=["Any","2+","3+","4+","5"], value="3+")
                ht_ci    = gr.Textbox(label="📅 Check-in", value=(datetime.date.today()+datetime.timedelta(days=30)).isoformat())
                ht_co    = gr.Textbox(label="📅 Check-out", value=(datetime.date.today()+datetime.timedelta(days=33)).isoformat())
            ht_btn = gr.Button("🔍 Find Hotels", variant="primary", size="lg")
            ht_out = gr.Markdown(value="*Search any city — AI finds real hotels with current prices!*")
            ht_btn.click(fn=lambda d,p,s,ci,co: search_hotels_ai(d,int(p) if p else 15000,s,ci,co),
                        inputs=[ht_dest,ht_price,ht_stars,ht_ci,ht_co], outputs=ht_out)

        # ── TAB 4: FLIGHTS ─────────────────────────────────────────────────
        with gr.Tab("✈️ Find Flights"):
            gr.HTML('<div style="background:#EBF8FF;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #3182CE"><h3 style="margin:0;color:#2C5282">✈️ AI Flight Search — Any Route</h3></div>')
            with gr.Row():
                fl_from  = gr.Textbox(label="From", placeholder="e.g. Mumbai, Delhi...", value="Mumbai")
                fl_to    = gr.Textbox(label="To", placeholder="e.g. Goa, Dubai, Singapore...", value="Goa")
            with gr.Row():
                fl_date  = gr.Textbox(label="📅 Travel Date", placeholder="e.g. December 2025", value="December 2025")
                fl_class = gr.Dropdown(label="Class", choices=["Economy","Business","First Class","Premium Economy"], value="Economy")
            fl_btn = gr.Button("✈️ Search Flights", variant="primary", size="lg")
            fl_out = gr.Markdown(value="*Search any route — AI finds real airlines with price ranges!*")
            fl_btn.click(fn=lambda o,d,dt,c: search_flights_ai(o,d,dt,c),
                        inputs=[fl_from,fl_to,fl_date,fl_class], outputs=fl_out)

        # ── TAB 5: PACKAGES ────────────────────────────────────────────────
        with gr.Tab("📦 Packages"):
            gr.HTML('<div style="background:#FFF5F0;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #FF6B35"><h3 style="margin:0;color:#C05621">📦 AI Travel Packages — Any Destination!</h3></div>')
            with gr.Row():
                pk_dest  = gr.Textbox(label="🌍 Destination", placeholder="e.g. Kerala, Europe, Southeast Asia...", value="Kerala")
                pk_budget= gr.Number(label="💰 Max Budget/Person (Rs.)", value=30000)
            with gr.Row():
                pk_style = gr.Dropdown(label="🎭 Style", choices=["Any","Beach","Adventure","Cultural","Luxury","Relaxation","Romantic","Family","Honeymoon","Spiritual","Budget"], value="Any")
                pk_days  = gr.Slider(label="🗓️ Days", minimum=2, maximum=21, value=5, step=1)
            pk_btn = gr.Button("📦 Generate Packages", variant="primary", size="lg")
            pk_out = gr.Markdown(value="*AI generates real travel packages with actual hotel names and prices!*")
            pk_btn.click(fn=lambda d,b,s,dy: search_packages_ai(d,int(b) if b else 30000,s,int(dy)),
                        inputs=[pk_dest,pk_budget,pk_style,pk_days], outputs=pk_out)

        # ── TAB 6: RECOMMENDER ─────────────────────────────────────────────
        with gr.Tab("🤖 Recommender"):
            with gr.Row():
                with gr.Column():
                    rc_int   = gr.Textbox(label="✨ Your Interests", placeholder="beaches, temples, street food, skiing...", lines=2, value="beaches, local food, water sports")
                    with gr.Row():
                        rc_bud   = gr.Dropdown(label="💰 Budget", choices=["Budget (under Rs.20,000/day)","Mid-range (Rs.20,000-60,000/day)","Premium (Rs.60,000-1,50,000/day)","Luxury (Rs.1,50,000+/day)"], value="Mid-range (Rs.20,000-60,000/day)")
                        rc_seas  = gr.Dropdown(label="🗓️ Season", choices=["Jan-Mar","Apr-Jun","Jul-Sep","Oct-Dec","Flexible"], value="Oct-Dec")
                    with gr.Row():
                        rc_trav  = gr.Dropdown(label="👥 Traveling As", choices=["Solo","Couple","Family with Kids","Group of Friends","Backpackers","Seniors"], value="Couple")
                        rc_style = gr.Dropdown(label="🎭 Style", choices=["Adventure Seeker","Culture Explorer","Beach Lover","Luxury Traveler","Budget Backpacker","Foodie","Nature Lover","Spiritual Seeker","Wellness Traveler"], value="Beach Lover")
                    rc_btn   = gr.Button("🤖 Get Recommendations", variant="primary", size="lg")
                with gr.Column(scale=2):
                    rc_out   = gr.Markdown(value="*Fill preferences — AI recommends perfect destinations!*")
            rc_btn.click(fn=lambda i,b,s,t,st: recommend_destinations_ai(i,b,s,t,st),
                        inputs=[rc_int,rc_bud,rc_seas,rc_trav,rc_style], outputs=rc_out)

        # ── TAB 7: ITINERARY ───────────────────────────────────────────────
        with gr.Tab("🗺️ Itinerary"):
            with gr.Row():
                with gr.Column():
                    it_dest  = gr.Textbox(label="🌍 Destination", value="Bali, Indonesia")
                    it_days  = gr.Slider(label="📅 Days", minimum=1, maximum=21, value=7, step=1)
                    it_style = gr.Dropdown(label="🎭 Style", choices=["Adventure","Relaxation","Cultural","Luxury","Budget","Romantic","Family","Backpacking","Foodie","Spiritual"], value="Adventure")
                    it_int   = gr.Textbox(label="💫 Interests", placeholder="temples, cooking, hiking...", value="temples, rice terraces, surfing")
                    it_blvl  = gr.Dropdown(label="💰 Budget Level", choices=["Budget (Rs.2,000-4,000/day)","Mid-range (Rs.4,000-10,000/day)","Luxury (Rs.10,000+/day)"], value="Mid-range (Rs.4,000-10,000/day)")
                    it_btn   = gr.Button("📋 Generate Itinerary", variant="primary", size="lg")
                with gr.Column(scale=2):
                    it_out   = gr.Markdown(value="*AI creates a detailed minute-by-minute itinerary with real venues!*")
            it_btn.click(fn=lambda d,dy,s,i,b: generate_itinerary_ai(d,int(dy),s,i,b),
                        inputs=[it_dest,it_days,it_style,it_int,it_blvl], outputs=it_out)

        # ── TAB 8: AI CHAT ─────────────────────────────────────────────────
        with gr.Tab("💬 AI Chat"):
            gr.HTML('<div style="background:#EBF8FF;border-radius:8px;padding:10px;margin-bottom:8px"><p style="margin:0;color:#2C5282;font-size:13px"><b>💡 Ask ANYTHING:</b> "Best time to visit Iceland?" | "Cheap flights Bangalore to Bangkok?" | "Visa requirements for Dubai?" | "Hidden gems in Rajasthan?"</p></div>')
            chatbot = gr.Chatbot(label="WanderAI — AI Travel Assistant", height=440, bubble_full_width=False,
                                 value=[[None, "👋 **Hello! I'm WanderAI** — powered by Groq LLaMA 3.3!\n\n🌍 Ask me ANYTHING about travel:\n- Real hotel & flight prices\n- Visa requirements for any country\n- Budget planning for any destination\n- Best restaurants & experiences\n- Hidden gems & off-beat destinations\n\nWhat adventure shall we plan? ✈️"]])
            with gr.Row():
                ch_in   = gr.Textbox(label="", placeholder="Ask anything about travel...", scale=5)
                ch_send = gr.Button("Send 🚀", variant="primary", scale=1)
                ch_clr  = gr.Button("🔄 Clear", scale=1)
            gr.HTML("<p style='color:#718096;font-size:12px;margin:4px 0'>Quick questions:</p>")
            with gr.Row():
                qb1 = gr.Button("🏔️ Best places in India for December", size="sm")
                qb2 = gr.Button("💰 Budget trip to Europe from India", size="sm")
                qb3 = gr.Button("🌏 Visa-free countries for Indians", size="sm")
                qb4 = gr.Button("🏨 Best luxury hotels in Maldives", size="sm")

            ch_send.click(fn=chat_ai, inputs=[ch_in,chatbot], outputs=[chatbot,ch_in])
            ch_in.submit(fn=chat_ai, inputs=[ch_in,chatbot], outputs=[chatbot,ch_in])
            ch_clr.click(fn=lambda: [[None,"Chat cleared! Ask me anything about travel! 🌍"]], outputs=chatbot)
            for btn, prompt in [
                (qb1,"Best places to visit in India in December with hotel prices"),
                (qb2,"How to plan a budget trip to Europe from India — visa, flights, costs"),
                (qb3,"Which countries can Indians visit without a visa in 2025?"),
                (qb4,"Best luxury overwater resorts in Maldives with current prices")]:
                btn.click(fn=lambda h,p=prompt: chat_ai(p,h), inputs=[chatbot], outputs=[chatbot,ch_in])

        # ── TAB 9: BOOK HOTEL ──────────────────────────────────────────────
        with gr.Tab("🎫 Book Hotel"):
            gr.HTML('<div style="background:#FAF5FF;border-radius:10px;padding:14px;margin-bottom:12px;border-left:4px solid #805AD5"><h3 style="margin:0;color:#553C9A">🎫 Hotel Booking — Type Any Hotel Name!</h3></div>')
            with gr.Row():
                bk_hotel = gr.Textbox(label="🏨 Hotel Name", placeholder="e.g. Taj Mahal Palace Mumbai, any hotel...")
                bk_dest  = gr.Textbox(label="📍 Destination", placeholder="e.g. Mumbai, Goa, Bali...")
            with gr.Row():
                bk_ci    = gr.Textbox(label="📅 Check-in", value=(datetime.date.today()+datetime.timedelta(days=30)).isoformat())
                bk_co    = gr.Textbox(label="📅 Check-out", value=(datetime.date.today()+datetime.timedelta(days=33)).isoformat())
            with gr.Row():
                bk_rooms = gr.Slider(label="🛏️ Rooms", minimum=1, maximum=5, value=1, step=1)
                bk_guests= gr.Slider(label="👥 Guests", minimum=1, maximum=10, value=2, step=1)
            gr.HTML("<hr>")
            with gr.Row():
                bk_name  = gr.Textbox(label="👤 Full Name")
                bk_email = gr.Textbox(label="📧 Email")
            bk_btn   = gr.Button("✅ Confirm Booking (Simulated)", variant="primary", size="lg")
            bk_rcpt  = gr.Textbox(label="📋 Booking Confirmation", lines=22, interactive=False)
            bk_btn.click(fn=book_hotel_ai,
                        inputs=[bk_hotel,bk_dest,bk_ci,bk_co,bk_rooms,bk_guests,bk_name,bk_email],
                        outputs=bk_rcpt)

        # ── TAB 10: BUDGET AI ──────────────────────────────────────────────
        with gr.Tab("💰 Budget AI"):
            with gr.Row():
                with gr.Column():
                    ba_dest  = gr.Textbox(label="🌍 Destination", value="Goa")
                    ba_total = gr.Number(label="💰 Total Budget (Rs.)", value=30000)
                    with gr.Row():
                        ba_days  = gr.Slider(label="🗓️ Days", minimum=1, maximum=21, value=5, step=1)
                        ba_trav  = gr.Number(label="👥 Travelers", value=2)
                    ba_btn   = gr.Button("💡 Optimize Budget", variant="primary", size="lg")
                    ba_info  = gr.HTML()
                with gr.Column(scale=2):
                    ba_out   = gr.Markdown(value="*AI creates a real optimized budget with actual prices!*")

            def do_budget(dest, total, days, trav):
                pd_ = int(total)//int(days)
                pp  = int(total)//int(trav)
                info = f"""<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin:8px 0">
<div style="background:white;border:1px solid #E2E8F0;border-radius:8px;padding:12px;text-align:center"><div style="font-size:20px">💰</div><div style="font-size:18px;font-weight:700;color:#38A169">Rs.{int(total):,}</div><div style="color:#718096;font-size:12px">Total Budget</div></div>
<div style="background:white;border:1px solid #E2E8F0;border-radius:8px;padding:12px;text-align:center"><div style="font-size:20px">📅</div><div style="font-size:18px;font-weight:700;color:#3182CE">Rs.{pd_:,}</div><div style="color:#718096;font-size:12px">Per Day</div></div>
<div style="background:white;border:1px solid #E2E8F0;border-radius:8px;padding:12px;text-align:center"><div style="font-size:20px">👤</div><div style="font-size:18px;font-weight:700;color:#E53E3E">Rs.{pp:,}</div><div style="color:#718096;font-size:12px">Per Person</div></div>
<div style="background:white;border:1px solid #E2E8F0;border-radius:8px;padding:12px;text-align:center"><div style="font-size:20px">🗓️</div><div style="font-size:18px;font-weight:700;color:#805AD5">{int(days)} days</div><div style="color:#718096;font-size:12px">Duration</div></div>
</div>"""
                return info, optimize_budget_ai(dest, int(total), int(days), int(trav))
            ba_btn.click(fn=do_budget, inputs=[ba_dest,ba_total,ba_days,ba_trav], outputs=[ba_info,ba_out])

        # ── TAB 11: AI INSIGHTS ────────────────────────────────────────────
        with gr.Tab("🔬 AI Insights"):
            with gr.Tabs():
                with gr.Tab("📊 Crowd Prediction"):
                    with gr.Row():
                        cr_dest = gr.Textbox(label="Destination", value="Bali")
                        cr_date = gr.Textbox(label="Travel Period", value="December 2025")
                    cr_out = gr.Markdown()
                    gr.Button("📊 Predict Crowds", variant="primary").click(fn=crowd_prediction_ai, inputs=[cr_dest,cr_date], outputs=cr_out)

                with gr.Tab("🌤️ Weather"):
                    with gr.Row():
                        wt_dest  = gr.Textbox(label="Destination", value="Kerala")
                        wt_month = gr.Dropdown(label="Month", choices=["January","February","March","April","May","June","July","August","September","October","November","December"], value="October")
                    wt_out = gr.Markdown()
                    gr.Button("🌤️ Weather Forecast", variant="primary").click(fn=weather_ai, inputs=[wt_dest,wt_month], outputs=wt_out)

                with gr.Tab("⚠️ Safety"):
                    with gr.Row():
                        sf_dest = gr.Textbox(label="Destination", value="Tokyo")
                        sf_date = gr.Textbox(label="Travel Date", value="March 2026")
                    sf_out = gr.Markdown()
                    gr.Button("🔒 Safety Report", variant="primary").click(fn=safety_ai, inputs=[sf_dest,sf_date], outputs=sf_out)

                with gr.Tab("🙏 Culture"):
                    cu_dest = gr.Textbox(label="Destination", value="Japan")
                    cu_out  = gr.Markdown()
                    gr.Button("🙏 Cultural Guide", variant="primary").click(fn=cultural_guide_ai, inputs=cu_dest, outputs=cu_out)

                with gr.Tab("🍽️ Food"):
                    fd_dest = gr.Textbox(label="Destination", value="Goa")
                    fd_out  = gr.Markdown()
                    gr.Button("🍽️ Food Guide", variant="primary").click(fn=food_guide_ai, inputs=fd_dest, outputs=fd_out)

                with gr.Tab("⚖️ Compare Cities"):
                    with gr.Row():
                        cmp_c1 = gr.Textbox(label="City 1", value="Goa")
                        cmp_c2 = gr.Textbox(label="City 2", value="Bali")
                    cmp_aspect = gr.Dropdown(label="Compare By",
                        choices=["Overall (Budget, Safety, Fun, Food, Culture)",
                                 "Budget & Cost of Living","Safety & Ease of Travel",
                                 "Food & Nightlife","Culture & Heritage",
                                 "Beaches & Nature","Best for Families","Best for Solo Travel"],
                        value="Overall (Budget, Safety, Fun, Food, Culture)")
                    cmp_out = gr.Markdown()
                    gr.Button("⚖️ Compare Now", variant="primary").click(
                        fn=lambda c1,c2,a: ask_ai(f"Compare {c1} vs {c2} for travelers focusing on {a}. Create a detailed comparison table and give a final verdict. Include real current prices in INR."),
                        inputs=[cmp_c1,cmp_c2,cmp_aspect], outputs=cmp_out)

        # ── TAB 12: WISHLIST ───────────────────────────────────────────────
        with gr.Tab("❤️ Wishlist"):
            wl_email = gr.Textbox(label="📧 Your Email")
            with gr.Row():
                with gr.Column():
                    gr.HTML("<h4>➕ Add to Wishlist</h4>")
                    wl_type  = gr.Dropdown(label="Type", choices=["Destination","Hotel","Package","Activity","Restaurant"], value="Destination")
                    wl_item  = gr.Textbox(label="Name", placeholder="e.g. Bali, Taj Hotel Goa, Kerala Package...")
                    wl_add   = gr.Button("❤️ Add", variant="primary")
                    wl_stat  = gr.HTML()
                with gr.Column():
                    gr.HTML("<h4>📋 My Wishlist</h4>")
                    wl_view  = gr.Button("🔍 View My Wishlist")
                    wl_out   = gr.HTML()
            wl_add.click(fn=add_wishlist, inputs=[wl_email,wl_type,wl_item], outputs=wl_stat)
            wl_view.click(fn=view_wishlist, inputs=wl_email, outputs=wl_out)

        # ── TAB 13: DASHBOARD ──────────────────────────────────────────────
        with gr.Tab("👤 Dashboard"):
            with gr.Tabs():
                with gr.Tab("🔐 Account"):
                    with gr.Row():
                        with gr.Column():
                            gr.HTML("<h3>🆕 Create Account</h3>")
                            rg_name = gr.Textbox(label="Full Name")
                            rg_email= gr.Textbox(label="Email")
                            rg_pw   = gr.Textbox(label="Password", type="password")
                            rg_btn  = gr.Button("🚀 Create Account", variant="primary")
                            rg_stat = gr.HTML()
                        with gr.Column():
                            gr.HTML("<h3>🔑 Sign In</h3>")
                            lg_email= gr.Textbox(label="Email")
                            lg_pw   = gr.Textbox(label="Password", type="password")
                            lg_btn  = gr.Button("🔑 Sign In", variant="primary")
                            lg_stat = gr.HTML()
                            lg_prof = gr.HTML()
                    rg_btn.click(fn=create_account, inputs=[rg_name,rg_email,rg_pw], outputs=rg_stat)
                    lg_btn.click(fn=login, inputs=[lg_email,lg_pw], outputs=[lg_stat,lg_prof])

                with gr.Tab("📋 My Bookings"):
                    mb_email = gr.Textbox(label="Your Email")
                    mb_btn   = gr.Button("🔍 Find My Bookings", variant="primary")
                    mb_out   = gr.HTML()
                    mb_btn.click(fn=get_bookings, inputs=mb_email, outputs=mb_out)

                with gr.Tab("🌍 Ask AI Anything"):
                    free_q   = gr.Textbox(label="Ask any travel question", placeholder="What are cheapest countries from India? Best hill stations in summer?...", lines=3)
                    free_btn = gr.Button("🤖 Get AI Answer", variant="primary")
                    free_out = gr.Markdown()
                    free_btn.click(fn=lambda q: ask_ai(q), inputs=free_q, outputs=free_out)

    gr.HTML("""<div style="margin-top:16px;padding:14px;background:#1A202C;border-radius:12px;text-align:center">
  <p style="margin:0;color:#A0AEC0;font-size:13px">🌍 <b style="color:white">WanderAI</b> — AI Tourism Platform | Powered by <b style="color:#F6AD55">Groq</b> + <b style="color:#3182CE">LLaMA 3.3 70B</b> | ♾️ Any Destination Worldwide</p>
  <p style="margin:4px 0 0;color:#718096;font-size:11px">⚠️ Bookings are simulated. Prices are AI estimates — verify on MakeMyTrip/Booking.com before booking.</p>
</div>""")

# ── LAUNCH ────────────────────────────────────────────────────────────────────
import subprocess, time
subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True)
time.sleep(2)
print("🚀 Launching WanderAI with Groq LLaMA 3.3...")
demo.launch(share=True, server_port=7860, server_name="0.0.0.0")

/tmp/ipykernel_1025/1911924586.py:629: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🌍 WanderAI — AI Tourism Platform", css=css,
/tmp/ipykernel_1025/1911924586.py:629: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="🌍 WanderAI — AI Tourism Platform", css=css,
/tmp/ipykernel_1025/1911924586.py:763: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="WanderAI — AI Travel Assistant", height=440, bubble_full_width=False,
/tmp/ipykernel_1025/1911924586.p

🚀 Launching WanderAI with Groq LLaMA 3.3...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b8da8149289287626.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
